<a href="https://colab.research.google.com/github/noorinbinary/delora/blob/main/Week1_MNLI_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login(new_session=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Meta-Llama-3-8B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Device set to use cuda:0


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 548.12 MiB is free. Process 2699 has 14.20 GiB memory in use. Of the allocated memory 13.98 GiB is allocated by PyTorch, and 129.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

KeyboardInterrupt: 

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded successfully!")
print(model)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully!
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    

In [ ]:
!nvidia-smi


Mon Nov  3 10:34:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             29W /   70W |    7106MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.38.1 datasets==2.19.1 peft==0.11.1 bitsandbytes==0.43.1 accelerate==0.30.1 trl==0.9.4 -q

Found existing installation: transformers 4.38.1
Uninstalling transformers-4.38.1:
  Successfully uninstalled transformers-4.38.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.38.1 which is incompatible.


In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os



In [ ]:

%%writefile train.py

import argparse
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ---
# DATA FORMATTING FUNCTIONS
# ---

def format_mnli_prompt(example):
    """
    Formats an MNLI example into an instruction-following prompt for Llama-3.
    """
    labels = ["entailment", "neutral", "contradiction"]
    id2label = {i: label for i, label in enumerate(labels)}

    prompt = f"""You are an expert in Natural Language Inference. Determine the relationship between the premise and the hypothesis.
The relationship can be: entailment, neutral, or contradiction.

### Premise:
{example['premise']}

### Hypothesis:
{example['hypothesis']}

### Answer:
{id2label[example['label']]}"""
    return {"text": prompt}

def format_samsum_prompt(example):
    """
    Formats a SAMSum example into an instruction-following prompt for Llama-3.
    """
    prompt = f"""You are an expert dialogue summarizer. Summarize the following conversation.

### Conversation:
{example['dialogue']}

### Summary:
{example['summary']}"""
    return {"text": prompt}

# ---
# MAIN EXECUTION FUNCTION
# ---

def main(args):
    """
    The main training function, parameterized by command-line arguments.
    """
    print(f"--- Starting training run for {args.output_dir} ---")

    # 1. --- Load Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print("Tokenizer loaded.")

    # 2. --- Load Dataset & Apply Formatting ---
    if args.dataset_name == "nyu-mll/glue":
        dataset = load_dataset(args.dataset_name, args.dataset_config)
        format_function = format_mnli_prompt
        print(f"Loading MNLI dataset.")
    elif args.dataset_name == "samsum":
        dataset = load_dataset(args.dataset_name)
        format_function = format_samsum_prompt
        print(f"Loading SAMSum dataset.")
    else:
        raise ValueError(f"Unsupported dataset: {args.dataset_name}")

    formatted_dataset = dataset.map(format_function)
    print("Dataset loaded and formatted.")

    # 3. --- QLoRA Configuration ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=False,
    )

    # 4. --- Load Base Model (Quantized) ---
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model)
    print("Base Llama-3 8B model loaded in 4-bit (QLoRA).")

    # 5. --- LoRA Configuration ---
    peft_config = LoraConfig(
        lora_alpha=16,
        lora_dropout=0.1,
        r=args.lora_rank,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    # 6. --- Training Arguments ---
    training_arguments = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        optim="paged_adamw_32bit",
        logging_steps=25,
        learning_rate=2e-4,
        fp16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type="constant",
        report_to="none",
    )

    # 7. --- Initialize Trainer ---
    trainer = SFTTrainer(
        model=model,
        peft_config=peft_config,
        train_dataset=formatted_dataset["train"],
        dataset_text_field="text",
        max_seq_length=1024,
        tokenizer=tokenizer,
        args=training_arguments,
        packing=True,
    )

    # 8. --- Train ---
    print(f"Starting training for {args.dataset_name}...")
    trainer.train()
    print("Training complete.")

    # 9. --- Save Final Adapter ---
    final_checkpoint_dir = os.path.join(args.output_dir, "final_checkpoint")
    trainer.model.save_pretrained(final_checkpoint_dir)
    tokenizer.save_pretrained(final_checkpoint_dir)
    print(f"Specialist adapter saved to {final_checkpoint_dir}")
    print("--- Training run complete. ---")

if __name__ == "__main__":
    # ---
    # ARGUMENT PARSING
    # This block only runs when the script is called from the command line (in Cell 3).
    # ---
    parser = argparse.ArgumentParser(description="Fine-tune a Llama model with LoRA on a specific task.")
    parser.add_argument("--model_name", type=str, default="meta-llama/Meta-Llama-3-8B-Instruct", help="The base LLM to fine-tune.")
    parser.add_argument("--dataset_name", type=str, required=True, help="Hugging Face dataset name (e.g., 'nyu-mll/glue' or 'samsum').")
    parser.add_argument("--dataset_config", type=str, default=None, help="Hugging Face dataset config (e.g., 'mnli').")
    parser.add_argument("--output_dir", type=str, required=True, help="Directory to save the training logs and final adapter.")
    parser.add_argument("--lora_rank", type=int, default=16, help="The rank 'r' for the LoRA matrices.")
    parser.add_argument("--num_epochs", type=int, default=1, help="Number of training epochs.")

    args = parser.parse_args()
    main(args)

print("'train.py' has been successfully written to the local filesystem.")

Overwriting train.py


In [ ]:
print("\nCELL 3: Launching training for Specialist 1 (MNLI)...")

!python train.py \
    --dataset_name "nyu-mll/glue" \
    --dataset_config "mnli" \
    --output_dir "./models/specialist_mnli" \
    --lora_rank 16 \
    --num_epochs 1

print("MNLI Specialist training complete.")


CELL 3: Launching training for Specialist 1 (MNLI)...
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2025-11-03 12:07:02.119201: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762171622.148270   37737 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762171622.157265   37737 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762171622.191679   37737 computation_placer.cc:177] computation placer already registered. Please check linkage and av

In [ ]:
print("\nCELL 4: Launching training for Specialist 2 (SAMSum)...")

!python train.py \
    --dataset_name "samsum" \
    --output_dir "./models/specialist_samsum" \
    --lora_rank 16 \
    --num_epochs 1

print("SAMSum Specialist training complete.")


CELL 4: Launching training for Specialist 2 (SAMSum)...
2025-11-03 12:10:51.907763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762171851.929279   38711 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762171851.935262   38711 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762171851.959794   38711 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762171851.959823   38711 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762171851.959829 